# Dictionary Views: Advanced Problems with Step-by-Step Solutions

In this notebook we are going to continue working with dictionary views:

- `keys`
- `values`
- `items`

The emphasis here is not just on getting an answer.

We are going to break each problem into small logical steps, inspect intermediate
results, discuss possible mistakes, and then build a final solution.

We will use dictionary views in situations such as:

- comparing API payloads
- merging layered configuration
- validating schemas
- finding exact shared items
- reconciling inventory
- analyzing permissions
- generating patches
- comparing many dictionaries at once

Before starting the problems, let's define a small helper that will make set
results easier to display.

Remember that set operations do not preserve dictionary insertion order, so
sorting is useful when we only want a predictable display.

In [1]:
def show_set(label, values):
    print(f"{label}: {sorted(values)}")

We will also use assertions throughout the notebook.

Assertions make the examples executable tests. If an assertion fails, Python
will stop and tell us that the result is not what we expected.

# Problem 1

## Detect changes in a live key view

Suppose we keep a reference to a dictionary's key view.

Later, the dictionary is changed by another part of the program.

We want to understand exactly what the retained view will show.

In [2]:
settings = {
    "theme": "dark",
    "language": "en",
    "page_size": 20,
}

keys_view = settings.keys()

At this point, the view reflects the current keys.

In [3]:
print(keys_view)

dict_keys(['theme', 'language', 'page_size'])


Now let's create a separate snapshot as a list.

Unlike the view, the list contains its own copy of the keys at this moment.

In [4]:
keys_snapshot = list(settings.keys())

print("view:", keys_view)
print("snapshot:", keys_snapshot)

view: dict_keys(['theme', 'language', 'page_size'])
snapshot: ['theme', 'language', 'page_size']


Next, we will add one key and remove another key.

In [5]:
settings["timezone"] = "UTC"
del settings["page_size"]

Let's inspect both objects again.

In [6]:
print("view:", keys_view)
print("snapshot:", keys_snapshot)

view: dict_keys(['theme', 'language', 'timezone'])
snapshot: ['theme', 'language', 'page_size']


The view changed automatically because it is connected to the dictionary.

The snapshot did not change because it is an independent list.

In [7]:
assert list(keys_view) == ["theme", "language", "timezone"]
assert keys_snapshot == ["theme", "language", "page_size"]

## Final solution

Use a view when you intentionally want a live representation.

Use `list(...)`, `tuple(...)`, or `set(...)` when you intentionally want a
snapshot.

# Problem 2

## Compare two API payloads by key

Suppose an API returned an old payload and a new payload.

We want to identify:

- fields that were added
- fields that were removed
- fields that still exist in both payloads

In [8]:
old_payload = {
    "id": 101,
    "name": "Ada",
    "email": "ada@example.com",
    "active": True,
}

new_payload = {
    "id": 101,
    "name": "Ada Lovelace",
    "active": True,
    "role": "admin",
}

The first step is to obtain the key views.

In [9]:
old_keys = old_payload.keys()
new_keys = new_payload.keys()

print(old_keys)
print(new_keys)

dict_keys(['id', 'name', 'email', 'active'])
dict_keys(['id', 'name', 'active', 'role'])


Fields added to the new payload are keys in `new_keys` but not in `old_keys`.

In [10]:
added_keys = new_keys - old_keys
show_set("added", added_keys)

added: ['role']


Fields removed from the new payload are keys in `old_keys` but not in `new_keys`.

In [11]:
removed_keys = old_keys - new_keys
show_set("removed", removed_keys)

removed: ['email']


Fields present in both payloads are found with an intersection.

In [12]:
common_keys = old_keys & new_keys
show_set("common", common_keys)

common: ['active', 'id', 'name']


Now we can combine those intermediate results into one report.

In [13]:
key_report = {
    "added": sorted(added_keys),
    "removed": sorted(removed_keys),
    "common": sorted(common_keys),
}

print(key_report)

{'added': ['role'], 'removed': ['email'], 'common': ['active', 'id', 'name']}


In [14]:
assert key_report == {
    "added": ["role"],
    "removed": ["email"],
    "common": ["active", "id", "name"],
}

## Important observation

This report compares only key presence.

It does not yet tell us whether values changed.

We will add value comparison in the next problem.

# Problem 3

## Separate unchanged fields from changed fields

We will continue using the API payloads from the previous problem.

Among the common keys, some values stayed the same and some values changed.

We already know how to find the common keys.

In [15]:
common_keys = old_payload.keys() & new_payload.keys()
show_set("common", common_keys)

common: ['active', 'id', 'name']


Let's inspect the values for each common key.

In [16]:
for key in sorted(common_keys):
    print(
        key,
        "old =", old_payload[key],
        "new =", new_payload[key],
    )

active old = True new = True
id old = 101 new = 101
name old = Ada new = Ada Lovelace


Now we can divide the common keys into two groups.

First, the unchanged fields.

In [17]:
unchanged_keys = {
    key
    for key in common_keys
    if old_payload[key] == new_payload[key]
}

show_set("unchanged", unchanged_keys)

unchanged: ['active', 'id']


Next, the changed fields.

In [18]:
changed_keys = {
    key
    for key in common_keys
    if old_payload[key] != new_payload[key]
}

show_set("changed", changed_keys)

changed: ['name']


For changed fields, a key alone is not enough.

A more useful result includes the value before and after the change.

In [19]:
changed_details = {
    key: {
        "before": old_payload[key],
        "after": new_payload[key],
    }
    for key in old_payload
    if key in changed_keys
}

print(changed_details)

{'name': {'before': 'Ada', 'after': 'Ada Lovelace'}}


In [20]:
assert unchanged_keys == {"id", "active"}
assert changed_keys == {"name"}
assert changed_details == {
    "name": {
        "before": "Ada",
        "after": "Ada Lovelace",
    }
}

## Why did we iterate over `old_payload` instead of `changed_keys`?

`changed_keys` is a set, so its order is not guaranteed.

Iterating over the original dictionary preserves the original field order.

# Problem 4

## Merge three configuration layers

Suppose an application loads configuration from three layers:

1. defaults
2. a file
3. environment variables

Later layers should override earlier layers.

In [21]:
defaults = {
    "host": "localhost",
    "port": 8000,
    "debug": False,
    "timeout": 30,
}

file_config = {
    "port": 8080,
    "timeout": 60,
    "log_level": "INFO",
}

environment = {
    "debug": True,
    "timeout": 10,
}

Before merging, let's inspect which keys overlap between the defaults and the
file configuration.

In [22]:
defaults_file_overlap = defaults.keys() & file_config.keys()
show_set("defaults/file overlap", defaults_file_overlap)

defaults/file overlap: ['port', 'timeout']


Now let's inspect which keys overlap between the partially merged configuration
and the environment layer.

First we create the partial merge.

In [23]:
partial = dict(defaults)
partial.update(file_config)

print(partial)

{'host': 'localhost', 'port': 8080, 'debug': False, 'timeout': 60, 'log_level': 'INFO'}


In [24]:
partial_environment_overlap = partial.keys() & environment.keys()
show_set("partial/environment overlap", partial_environment_overlap)

partial/environment overlap: ['debug', 'timeout']


The final merge can now be created.

In [25]:
final_config = dict(defaults)
final_config.update(file_config)
final_config.update(environment)

print(final_config)

{'host': 'localhost', 'port': 8080, 'debug': True, 'timeout': 10, 'log_level': 'INFO'}


Let's verify the precedence rules:

- `port` comes from the file
- `debug` comes from the environment
- `timeout` comes from the environment
- `host` remains from defaults
- `log_level` is introduced by the file

In [26]:
assert final_config == {
    "host": "localhost",
    "port": 8080,
    "debug": True,
    "timeout": 10,
    "log_level": "INFO",
}

## Advanced question

Which keys were overridden at least once?

A key was overridden when it appeared in more than one layer.

We can count how many layers contain each key.

In [27]:
from collections import Counter

layer_key_counts = Counter(
    key
    for layer in (defaults, file_config, environment)
    for key in layer.keys()
)

print(layer_key_counts)

Counter({'timeout': 3, 'port': 2, 'debug': 2, 'host': 1, 'log_level': 1})


In [28]:
overridden_keys = {
    key
    for key, count in layer_key_counts.items()
    if count > 1
}

show_set("overridden", overridden_keys)

overridden: ['debug', 'port', 'timeout']


In [29]:
assert overridden_keys == {"port", "debug", "timeout"}

# Problem 5

## Find exact shared items

Two dictionaries may share a key but store different values.

An exact shared item requires both the key and the value to match.

In [30]:
inventory_a = {
    "pen": 10,
    "paper": 25,
    "folder": 8,
}

inventory_b = {
    "paper": 25,
    "folder": 12,
    "stapler": 4,
}

First, let's find the shared keys.

In [31]:
shared_keys = inventory_a.keys() & inventory_b.keys()
show_set("shared keys", shared_keys)

shared keys: ['folder', 'paper']


The key `folder` is shared, but its values are different.

The key `paper` is shared and its values are equal.

Because the values are integers and therefore hashable, we can perform a set
operation on the item views.

In [32]:
shared_items = inventory_a.items() & inventory_b.items()
print(shared_items)

{('paper', 25)}


In [33]:
assert shared_items == {("paper", 25)}

Now let's inspect the items that occur only in the first dictionary.

In [34]:
only_in_a_items = inventory_a.items() - inventory_b.items()
print(only_in_a_items)

{('pen', 10), ('folder', 8)}


Notice that `("folder", 8)` is considered different from `("folder", 12)`.

The key is the same, but the complete item tuple is different.

In [35]:
assert only_in_a_items == {
    ("pen", 10),
    ("folder", 8),
}

# Problem 6

## Handle unhashable values safely

Now suppose dictionary values are lists.

In [36]:
teams_a = {
    "red": ["Ana", "Bo"],
    "blue": ["Cy", "Dee"],
}

teams_b = {
    "blue": ["Cy", "Dee"],
    "green": ["Eli"],
}

The key views still support set operations because dictionary keys must be
hashable.

In [37]:
show_set("shared team names", teams_a.keys() & teams_b.keys())

shared team names: ['blue']


However, item-view set operations need the entire `(key, value)` tuple to be
hashable.

A tuple containing a list is not hashable.

In [38]:
try:
    teams_a.items() & teams_b.items()
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

TypeError: unhashable type: 'list'


We still want to find shared keys whose values are equal.

We can solve this by intersecting the keys first and comparing values normally.

In [39]:
shared_equal_team_keys = {
    key
    for key in teams_a.keys() & teams_b.keys()
    if teams_a[key] == teams_b[key]
}

show_set("shared keys with equal values", shared_equal_team_keys)

shared keys with equal values: ['blue']


In [40]:
assert shared_equal_team_keys == {"blue"}

## General solution

In [41]:
def shared_equal_keys(left, right):
    common = left.keys() & right.keys()
    return {
        key
        for key in common
        if left[key] == right[key]
    }

In [42]:
assert shared_equal_keys(teams_a, teams_b) == {"blue"}
assert shared_equal_keys(
    {"x": {"enabled": True}},
    {"x": {"enabled": True}},
) == {"x"}

This solution works with hashable and unhashable values because equality does
not require hashing.

# Problem 7

## Remove invalid records safely

Suppose a dictionary maps user names to scores.

We want to remove every user with a negative score.

In [43]:
scores = {
    "Ana": 12,
    "Bo": -1,
    "Cy": 8,
    "Dee": -4,
}

A tempting solution is to delete keys while directly iterating over the
dictionary.

In [44]:
unsafe_scores = dict(scores)

try:
    for name in unsafe_scores:
        if unsafe_scores[name] < 0:
            del unsafe_scores[name]
except RuntimeError as exc:
    print(type(exc).__name__ + ":", exc)

RuntimeError: dictionary changed size during iteration


The dictionary changed size during iteration, so Python raised an error.

One safe solution is to create a snapshot of the items.

In [45]:
safe_scores_1 = dict(scores)

for name, score in list(safe_scores_1.items()):
    if score < 0:
        del safe_scores_1[name]

print(safe_scores_1)

{'Ana': 12, 'Cy': 8}


Another safe solution is to collect the keys first and delete them afterward.

In [46]:
safe_scores_2 = dict(scores)

names_to_delete = [
    name
    for name, score in safe_scores_2.items()
    if score < 0
]

print("to delete:", names_to_delete)

to delete: ['Bo', 'Dee']


In [47]:
for name in names_to_delete:
    del safe_scores_2[name]

print(safe_scores_2)

{'Ana': 12, 'Cy': 8}


In [48]:
assert safe_scores_1 == {"Ana": 12, "Cy": 8}
assert safe_scores_2 == {"Ana": 12, "Cy": 8}

## Can we update existing values during iteration?

Updating an existing key does not change dictionary size.

In [49]:
adjusted_scores = dict(scores)

for name in adjusted_scores:
    adjusted_scores[name] = max(adjusted_scores[name], 0)

print(adjusted_scores)

{'Ana': 12, 'Bo': 0, 'Cy': 8, 'Dee': 0}


In [50]:
assert adjusted_scores == {
    "Ana": 12,
    "Bo": 0,
    "Cy": 8,
    "Dee": 0,
}

# Problem 8

## Validate a versioned payload schema

Suppose version 2 of an API requires these fields:

In [51]:
required_v2 = {
    "id",
    "name",
    "email",
    "created_at",
}

optional_v2 = {
    "phone",
    "timezone",
    "language",
}

A valid payload must:

- contain every required field
- contain no field outside the required and optional sets

In [52]:
payload = {
    "id": 501,
    "name": "Mira",
    "email": "mira@example.com",
    "timezone": "Europe/Sofia",
    "debug": True,
}

First, let's find missing required fields.

In [53]:
missing_required = required_v2 - payload.keys()
show_set("missing required", missing_required)

missing required: ['created_at']


Next, let's build the complete allowed-key set.

In [54]:
allowed_v2 = required_v2 | optional_v2
show_set("allowed", allowed_v2)

allowed: ['created_at', 'email', 'id', 'language', 'name', 'phone', 'timezone']


Unexpected fields are payload keys that are not allowed.

In [55]:
unexpected = payload.keys() - allowed_v2
show_set("unexpected", unexpected)

unexpected: ['debug']


Now we can build a reusable validator.

In [56]:
def validate_payload(payload, required, optional=()):
    required = set(required)
    allowed = required | set(optional)

    missing = required - payload.keys()
    unexpected = payload.keys() - allowed

    return {
        "valid": not missing and not unexpected,
        "missing": sorted(missing),
        "unexpected": sorted(unexpected),
    }

In [57]:
validation = validate_payload(payload, required_v2, optional_v2)
print(validation)

{'valid': False, 'missing': ['created_at'], 'unexpected': ['debug']}


In [58]:
assert validation == {
    "valid": False,
    "missing": ["created_at"],
    "unexpected": ["debug"],
}

We can also use subset notation for a quick required-field check.

In [59]:
has_all_required = required_v2 <= payload.keys()
print(has_all_required)

assert has_all_required is False

False


# Problem 9

## Build a migration plan between two schemas

Suppose a database table is moving from schema version 1 to schema version 2.

We want to identify:

- columns to add
- columns to remove
- columns that remain

In [60]:
schema_v1 = {
    "id": "INTEGER",
    "full_name": "TEXT",
    "email": "TEXT",
    "active": "BOOLEAN",
}

schema_v2 = {
    "id": "INTEGER",
    "first_name": "TEXT",
    "last_name": "TEXT",
    "email": "VARCHAR(320)",
    "active": "BOOLEAN",
}

The columns to add are found with right difference.

In [61]:
columns_to_add = schema_v2.keys() - schema_v1.keys()
show_set("add", columns_to_add)

add: ['first_name', 'last_name']


The columns to remove are found with left difference.

In [62]:
columns_to_remove = schema_v1.keys() - schema_v2.keys()
show_set("remove", columns_to_remove)

remove: ['full_name']


The shared columns are found with intersection.

In [63]:
shared_columns = schema_v1.keys() & schema_v2.keys()
show_set("shared", shared_columns)

shared: ['active', 'email', 'id']


Among the shared columns, data types may have changed.

In [64]:
type_changes = {
    column: {
        "before": schema_v1[column],
        "after": schema_v2[column],
    }
    for column in schema_v1
    if (
        column in shared_columns
        and schema_v1[column] != schema_v2[column]
    )
}

print(type_changes)

{'email': {'before': 'TEXT', 'after': 'VARCHAR(320)'}}


Now let's create an ordered migration plan.

In [65]:
migration_plan = {
    "remove": [
        column
        for column in schema_v1
        if column in columns_to_remove
    ],
    "add": {
        column: schema_v2[column]
        for column in schema_v2
        if column in columns_to_add
    },
    "alter_type": type_changes,
}

print(migration_plan)

{'remove': ['full_name'], 'add': {'first_name': 'TEXT', 'last_name': 'TEXT'}, 'alter_type': {'email': {'before': 'TEXT', 'after': 'VARCHAR(320)'}}}


In [66]:
assert migration_plan == {
    "remove": ["full_name"],
    "add": {
        "first_name": "TEXT",
        "last_name": "TEXT",
    },
    "alter_type": {
        "email": {
            "before": "TEXT",
            "after": "VARCHAR(320)",
        }
    },
}

# Problem 10

## Compare permissions for two roles

Suppose permissions are stored as dictionary keys.

The dictionary values contain a description of each permission.

In [67]:
editor_permissions = {
    "read": "Read documents",
    "write": "Create and edit documents",
    "comment": "Add comments",
}

reviewer_permissions = {
    "read": "Read documents",
    "comment": "Add comments",
    "approve": "Approve documents",
}

Let's identify permissions available to both roles.

In [68]:
shared_permissions = (
    editor_permissions.keys()
    & reviewer_permissions.keys()
)

show_set("shared", shared_permissions)

shared: ['comment', 'read']


Now let's find permissions only the editor has.

In [69]:
editor_only = (
    editor_permissions.keys()
    - reviewer_permissions.keys()
)

show_set("editor only", editor_only)

editor only: ['write']


And permissions only the reviewer has.

In [70]:
reviewer_only = (
    reviewer_permissions.keys()
    - editor_permissions.keys()
)

show_set("reviewer only", reviewer_only)

reviewer only: ['approve']


Let's create a comparison report that also includes descriptions.

In [71]:
permission_report = {
    "shared": {
        permission: editor_permissions[permission]
        for permission in editor_permissions
        if permission in shared_permissions
    },
    "editor_only": {
        permission: editor_permissions[permission]
        for permission in editor_permissions
        if permission in editor_only
    },
    "reviewer_only": {
        permission: reviewer_permissions[permission]
        for permission in reviewer_permissions
        if permission in reviewer_only
    },
}

print(permission_report)

{'shared': {'read': 'Read documents', 'comment': 'Add comments'}, 'editor_only': {'write': 'Create and edit documents'}, 'reviewer_only': {'approve': 'Approve documents'}}


In [72]:
assert permission_report == {
    "shared": {
        "read": "Read documents",
        "comment": "Add comments",
    },
    "editor_only": {
        "write": "Create and edit documents",
    },
    "reviewer_only": {
        "approve": "Approve documents",
    },
}

# Problem 11

## Compare many dictionaries at once

Suppose three regional services publish feature dictionaries.

We want to know:

- features available everywhere
- features available somewhere
- features available in exactly one region

In [73]:
europe = {
    "search": True,
    "payments": True,
    "recommendations": True,
}

asia = {
    "search": True,
    "payments": True,
    "translations": True,
}

america = {
    "search": True,
    "recommendations": True,
    "live_chat": True,
}

To find keys present everywhere, we can intersect all three key sets.

In [74]:
everywhere = (
    europe.keys()
    & asia.keys()
    & america.keys()
)

show_set("everywhere", everywhere)

everywhere: ['search']


To find keys present somewhere, we can union all three key sets.

In [75]:
somewhere = (
    europe.keys()
    | asia.keys()
    | america.keys()
)

show_set("somewhere", somewhere)

somewhere: ['live_chat', 'payments', 'recommendations', 'search', 'translations']


Finding keys present in exactly one dictionary requires counting.

In [76]:
from collections import Counter

feature_counts = Counter(
    key
    for region in (europe, asia, america)
    for key in region.keys()
)

print(feature_counts)

Counter({'search': 3, 'payments': 2, 'recommendations': 2, 'translations': 1, 'live_chat': 1})


In [77]:
exactly_one_region = {
    key
    for key, count in feature_counts.items()
    if count == 1
}

show_set("exactly one region", exactly_one_region)

exactly one region: ['live_chat', 'translations']


In [78]:
assert everywhere == {"search"}
assert somewhere == {
    "search",
    "payments",
    "recommendations",
    "translations",
    "live_chat",
}
assert exactly_one_region == {
    "translations",
    "live_chat",
}

## Reusable function

In [79]:
def analyze_key_presence(*mappings):
    if not mappings:
        return {
            "everywhere": set(),
            "somewhere": set(),
            "exactly_one": set(),
        }

    key_sets = [set(mapping.keys()) for mapping in mappings]

    counts = Counter(
        key
        for key_set in key_sets
        for key in key_set
    )

    return {
        "everywhere": set.intersection(*key_sets),
        "somewhere": set.union(*key_sets),
        "exactly_one": {
            key
            for key, count in counts.items()
            if count == 1
        },
    }

In [80]:
analysis = analyze_key_presence(europe, asia, america)
assert analysis["everywhere"] == {"search"}
assert analysis["exactly_one"] == {"translations", "live_chat"}

# Problem 12

## Generate and apply a dictionary patch

Suppose we have an old configuration and a new configuration.

We want to generate a patch containing:

- keys to set
- keys to delete

In [81]:
old_config = {
    "host": "db-01",
    "port": 5432,
    "ssl": True,
    "timeout": 30,
}

new_config = {
    "host": "db-02",
    "port": 5432,
    "ssl": True,
    "pool_size": 20,
}

Keys that disappear must be deleted.

In [82]:
delete_keys = old_config.keys() - new_config.keys()
show_set("delete", delete_keys)

delete: ['timeout']


Keys that are new or whose values changed must be set.

Let's first identify new keys.

In [83]:
new_keys = new_config.keys() - old_config.keys()
show_set("new", new_keys)

new: ['pool_size']


Now let's identify changed shared keys.

In [84]:
shared_keys = old_config.keys() & new_config.keys()

changed_shared_keys = {
    key
    for key in shared_keys
    if old_config[key] != new_config[key]
}

show_set("changed shared", changed_shared_keys)

changed shared: ['host']


The complete set of keys to set is the union of new keys and changed shared
keys.

In [85]:
set_keys = new_keys | changed_shared_keys
show_set("set", set_keys)

set: ['host', 'pool_size']


Now we can build the patch.

We will preserve the order of the new configuration for the `set` section and
the order of the old configuration for the `delete` section.

In [86]:
patch = {
    "set": {
        key: new_config[key]
        for key in new_config
        if key in set_keys
    },
    "delete": [
        key
        for key in old_config
        if key in delete_keys
    ],
}

print(patch)

{'set': {'host': 'db-02', 'pool_size': 20}, 'delete': ['timeout']}


In [87]:
assert patch == {
    "set": {
        "host": "db-02",
        "pool_size": 20,
    },
    "delete": ["timeout"],
}

Now let's write a function that applies the patch.

In [88]:
def apply_patch(original, patch):
    result = dict(original)

    for key in patch["delete"]:
        result.pop(key, None)

    for key, value in patch["set"].items():
        result[key] = value

    return result

In [89]:
patched_config = apply_patch(old_config, patch)

print(patched_config)
print(new_config)

{'host': 'db-02', 'port': 5432, 'ssl': True, 'pool_size': 20}
{'host': 'db-02', 'port': 5432, 'ssl': True, 'pool_size': 20}


In [90]:
assert patched_config == new_config

# Problem 13

## Preserve order while using set relationships

Suppose two dictionaries contain task information.

We want all keys in this order:

1. every key from the first dictionary
2. then keys found only in the second dictionary

In [91]:
task_a = {
    "id": 1,
    "title": "Write report",
    "owner": "Ana",
    "status": "open",
}

task_b = {
    "status": "closed",
    "owner": "Ana",
    "completed_at": "2026-08-03",
    "reviewer": "Bo",
}

A direct union finds all keys.

In [92]:
union_keys = task_a.keys() | task_b.keys()
print(union_keys)

{'title', 'status', 'completed_at', 'reviewer', 'owner', 'id'}


But the union result is a set.

We should not use its order to build a user-facing result.

Let's begin with the keys from the first dictionary.

In [93]:
ordered_keys = list(task_a.keys())
print(ordered_keys)

['id', 'title', 'owner', 'status']


Now append only the unseen keys from the second dictionary.

In [94]:
ordered_keys.extend(
    key
    for key in task_b
    if key not in task_a
)

print(ordered_keys)

['id', 'title', 'owner', 'status', 'completed_at', 'reviewer']


In [95]:
assert ordered_keys == [
    "id",
    "title",
    "owner",
    "status",
    "completed_at",
    "reviewer",
]

## Reusable solution

In [96]:
def ordered_key_union(*mappings):
    seen = set()
    result = []

    for mapping in mappings:
        for key in mapping:
            if key not in seen:
                seen.add(key)
                result.append(key)

    return result

In [97]:
assert ordered_key_union(task_a, task_b) == ordered_keys
assert ordered_key_union(
    {"a": 1, "b": 2},
    {"b": 20, "c": 3},
    {"a": 10, "d": 4},
) == ["a", "b", "c", "d"]

# Problem 14

## Distinguish a missing key from a stored `None`

Suppose a user profile may store `None` as a real value.

We want to compare selected fields between two versions of the profile.

In [98]:
profile_before = {
    "name": "Lina",
    "email": None,
    "phone": "123",
}

profile_after = {
    "name": "Lina",
    "email": "lina@example.com",
    "timezone": None,
}

Using `get` without a special default creates ambiguity.

Both of these expressions return `None`, but for different reasons.

In [99]:
print(profile_before.get("email"))
print(profile_before.get("timezone"))

None
None


The first key exists and stores `None`.

The second key does not exist.

In [100]:
print("email exists:", "email" in profile_before)
print("timezone exists:", "timezone" in profile_before)

email exists: True
timezone exists: False


A unique sentinel gives us a value that cannot be confused with normal data.

In [101]:
MISSING = object()

before_timezone = profile_before.get("timezone", MISSING)
after_timezone = profile_after.get("timezone", MISSING)

print(before_timezone is MISSING)
print(after_timezone is MISSING)

True
False


Now we can build a correct comparison function.

In [102]:
def compare_fields(before, after, fields):
    missing = {}
    changed = {}
    unchanged = []

    sentinel = object()

    for field in fields:
        before_value = before.get(field, sentinel)
        after_value = after.get(field, sentinel)

        if before_value is sentinel or after_value is sentinel:
            missing[field] = {
                "missing_before": before_value is sentinel,
                "missing_after": after_value is sentinel,
            }
        elif before_value == after_value:
            unchanged.append(field)
        else:
            changed[field] = {
                "before": before_value,
                "after": after_value,
            }

    return {
        "missing": missing,
        "changed": changed,
        "unchanged": unchanged,
    }

In [103]:
profile_comparison = compare_fields(
    profile_before,
    profile_after,
    ["name", "email", "phone", "timezone"],
)

print(profile_comparison)

{'missing': {'phone': {'missing_before': False, 'missing_after': True}, 'timezone': {'missing_before': True, 'missing_after': False}}, 'changed': {'email': {'before': None, 'after': 'lina@example.com'}}, 'unchanged': ['name']}


In [104]:
assert profile_comparison == {
    "missing": {
        "phone": {
            "missing_before": False,
            "missing_after": True,
        },
        "timezone": {
            "missing_before": True,
            "missing_after": False,
        },
    },
    "changed": {
        "email": {
            "before": None,
            "after": "lina@example.com",
        },
    },
    "unchanged": ["name"],
}

# Problem 15

## Capstone: reconcile two product catalogs

Two systems contain product dictionaries.

For every product key, we want to determine whether the product is:

- only in the old catalog
- only in the new catalog
- unchanged
- changed

In [105]:
old_catalog = {
    "P100": {
        "name": "Keyboard",
        "price": 50,
        "active": True,
    },
    "P200": {
        "name": "Mouse",
        "price": 20,
        "active": True,
    },
    "P300": {
        "name": "Monitor",
        "price": 200,
        "active": False,
    },
}

new_catalog = {
    "P100": {
        "name": "Keyboard",
        "price": 55,
        "active": True,
    },
    "P200": {
        "name": "Mouse",
        "price": 20,
        "active": True,
    },
    "P400": {
        "name": "Webcam",
        "price": 80,
        "active": True,
    },
}

First, let's identify products found only in the old catalog.

In [106]:
old_only_products = old_catalog.keys() - new_catalog.keys()
show_set("old only", old_only_products)

old only: ['P300']


Next, products found only in the new catalog.

In [107]:
new_only_products = new_catalog.keys() - old_catalog.keys()
show_set("new only", new_only_products)

new only: ['P400']


Now, products found in both catalogs.

In [108]:
common_products = old_catalog.keys() & new_catalog.keys()
show_set("common", common_products)

common: ['P100', 'P200']


The product values are dictionaries.

That means we cannot safely use item-view set operations.

Instead, we compare values directly for the common product keys.

In [109]:
unchanged_products = {
    product_id
    for product_id in common_products
    if old_catalog[product_id] == new_catalog[product_id]
}

changed_products = {
    product_id
    for product_id in common_products
    if old_catalog[product_id] != new_catalog[product_id]
}

show_set("unchanged", unchanged_products)
show_set("changed", changed_products)

unchanged: ['P200']
changed: ['P100']


For changed products, let's compare individual fields.

In [110]:
def compare_product_fields(old_product, new_product):
    common_fields = old_product.keys() & new_product.keys()

    return {
        field: {
            "before": old_product[field],
            "after": new_product[field],
        }
        for field in old_product
        if (
            field in common_fields
            and old_product[field] != new_product[field]
        )
    }

In [111]:
product_changes = {
    product_id: compare_product_fields(
        old_catalog[product_id],
        new_catalog[product_id],
    )
    for product_id in old_catalog
    if product_id in changed_products
}

print(product_changes)

{'P100': {'price': {'before': 50, 'after': 55}}}


Finally, let's build the complete reconciliation result.

In [112]:
catalog_report = {
    "removed": {
        product_id: old_catalog[product_id]
        for product_id in old_catalog
        if product_id in old_only_products
    },
    "added": {
        product_id: new_catalog[product_id]
        for product_id in new_catalog
        if product_id in new_only_products
    },
    "unchanged": [
        product_id
        for product_id in old_catalog
        if product_id in unchanged_products
    ],
    "changed": product_changes,
}

print(catalog_report)

{'removed': {'P300': {'name': 'Monitor', 'price': 200, 'active': False}}, 'added': {'P400': {'name': 'Webcam', 'price': 80, 'active': True}}, 'unchanged': ['P200'], 'changed': {'P100': {'price': {'before': 50, 'after': 55}}}}


In [113]:
assert catalog_report == {
    "removed": {
        "P300": {
            "name": "Monitor",
            "price": 200,
            "active": False,
        }
    },
    "added": {
        "P400": {
            "name": "Webcam",
            "price": 80,
            "active": True,
        }
    },
    "unchanged": ["P200"],
    "changed": {
        "P100": {
            "price": {
                "before": 50,
                "after": 55,
            }
        }
    },
}

# Final review

The most important ideas from these problems are:

- dictionary views are live
- key views are set-like
- value views are not set-like
- item views are set-like only when complete items are hashable
- set-operation results do not preserve dictionary order
- source-dictionary iteration is useful for deterministic output
- dictionary size should not change during direct iteration
- key membership is different from value truthiness
- sentinels help distinguish missing keys from stored `None`

Let's run one final regression test using the reusable functions created in this
notebook.

In [114]:
def run_final_checks():
    assert validate_payload(
        {"id": 1},
        required={"id"},
    ) == {
        "valid": True,
        "missing": [],
        "unexpected": [],
    }

    assert shared_equal_keys(
        {"a": [1], "b": [2]},
        {"a": [1], "c": [3]},
    ) == {"a"}

    assert ordered_key_union(
        {"a": 1},
        {"a": 2, "b": 3},
        {"c": 4},
    ) == ["a", "b", "c"]

    assert apply_patch(
        {"a": 1, "b": 2},
        {
            "set": {"a": 10, "c": 3},
            "delete": ["b"],
        },
    ) == {
        "a": 10,
        "c": 3,
    }

    assert analyze_key_presence(
        {"a": 1, "b": 2},
        {"b": 3, "c": 4},
    ) == {
        "everywhere": {"b"},
        "somewhere": {"a", "b", "c"},
        "exactly_one": {"a", "c"},
    }

    return "All final checks passed."


run_final_checks()

'All final checks passed.'